# RE_201 · Bolets: comestible o verinós?

**Bloc 2 · Supervisat · repte 1 de 6 · durada: 1 o 2 sessions de 2 h**

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Fas de voluntari a una associació micològica. Volen una app que, a partir de trets que qualsevol
persona pot mirar al bosc (forma i color del barret, olor, làmines, anell...), digui si un bolet és
**comestible** o **verinós**. Abans de posar-hi el nom de l'associació, et demanen tres coses:

- **Es pot decidir amb un model?** Amb quin, i com de gran ha de ser?
- **Què ha après el model?** Ha de ser una cosa que una persona pugui seguir.
- **Què passa quan s'equivoca?** Aquí no tots els errors costen el mateix.

L'enunciat complet, amb el diccionari de columnes, és a [RE_201](../RE_201_bolets.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

## Què necessites saber abans

- Separar entrenament i test: [ML_00](../../00_demo/ML_00_demo_iris.ipynb) i [EX_00 §1](../../02_practica/EX_00_caixa_eines.ipynb)
- k veïns i la distància: [ML_01 §2-§4](../../01_teoria/ML_01_knn.ipynb)
- Gini, el tall i la profunditat d'un arbre: [ML_02 §2, §3 i §6](../../01_teoria/ML_02_arbres.ipynb)
- Columnes de text i `OneHotEncoder`: [EX_00 §4](../../02_practica/EX_00_caixa_eines.ipynb)
- Matriu de confusió i *recall*: [EX_00 §7](../../02_practica/EX_00_caixa_eines.ipynb) i [EX_02](../../02_practica/EX_02_cancer.ipynb)

## Les dades

**Mushroom** (UCI, 1987), a OpenML amb `data_id=24`. Llicència CC BY 4.0. Cada fila és una
**espècie de bolet descrita** a la guia *The Audubon Society Field Guide to North American
Mushrooms*, amb 22 trets visibles codificats amb una lletra, i la columna `class` (`e` = comestible,
*edible*; `p` = verinós o dubtós, *poisonous*). El significat de cada lletra és a la fitxa del
dataset: <https://www.openml.org/d/24> (o a la UCI: <https://archive.ics.uci.edu/dataset/73/mushroom>).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=24, as_frame=True).frame
print(df.shape)
df.head()

## Nivell 1 · Explorar

### 1.1 · Què hi ha a la taula

Forma, tipus de cada columna i quants valors diferents té cadascuna. Hi ha alguna columna que no pugui ajudar a decidir res?

In [ ]:
print(df.dtypes.value_counts())
print(df.nunique().sort_values())
print("Columnes amb un sol valor:", list(df.columns[df.nunique() == 1]))

> **Resultat.** Totes les columnes són categòriques. `veil-type` només té un valor: no pot fer cap tall, i es pot treure o deixar (el model l'ignorarà). Mireu també `bruises%3F`: el nom porta un `?` codificat.

### 1.2 · Els nuls

Quina columna té valors que falten, i quants? Mira a la fitxa del dataset què vol dir aquesta columna. Decideix (i escriu per què) si esborraràs files, la columna, o tractaràs el nul com una categoria més.

In [ ]:
nuls = df.isna().sum()
print(nuls[nuls > 0])
print((df.isna().mean()[nuls > 0] * 100).round(1))
print(pd.crosstab(df["stalk-root"].astype(object).fillna("falta"), df["class"]))

> **Resultat.** Només `stalk-root` té nuls (2.480 files, un 30,5 %). Esborrar files perd massa taula. Tractar el nul com a categoria pròpia és la decisió més senzilla i honesta (el crosstab mostra que el nul no es reparteix igual entre classes, i això ja és informació).

### 1.3 · Les classes

Quants bolets hi ha de cada classe? Si un model digués sempre la classe majoritària, quina precisió (*accuracy*) tindria?

In [ ]:
print(df["class"].value_counts())
print("Precisió del model ximple:", df["class"].value_counts(normalize=True).max().round(3))

> **Resultat.** 4.208 comestibles i 3.916 verinosos: gairebé equilibrat, la línia base és 0,518.

## Nivell 2 · Respondre

### 2.1 · De lletres a números, i la partició

Prepara `X` (els 22 trets, amb la decisió de l'1.2) i `y` (`class`). Parteix en entrenament i test (25 % de test, estratificat, `random_state=42`). Codifica `X` amb un `OneHotEncoder` **ajustat només amb l'entrenament**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

X = df.drop(columns="class").astype(object).fillna("falta")
y = df["class"].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
Xtr = ohe.fit_transform(X_train)
Xte = ohe.transform(X_test)
noms = ohe.get_feature_names_out()

print(Xtr.shape, Xte.shape)
assert len(X_train) + len(X_test) == len(df)
assert not np.isnan(Xtr).any()
assert (Xtr.sum(axis=1) == 22).all()

> **Resultat.** Surten 6.093 files d'entrenament i 2.031 de test, amb 117 columnes 0/1 (amb el nul com a categoria).

### 2.2 · Quina mida d'arbre cal

Entrena arbres de profunditat 1 a 10 i dibuixa la precisió d'entrenament i de test segons la profunditat. Quina és la profunditat **més petita** que ja dona el millor resultat de test? Per què t'interessa la més petita i no la més gran?

In [ ]:
from sklearn.tree import DecisionTreeClassifier

prof = list(range(1, 11))
tr, te = [], []
for d in prof:
    a = DecisionTreeClassifier(max_depth=d, random_state=42).fit(Xtr, y_train)
    assert a.get_depth() <= d
    tr.append(a.score(Xtr, y_train))
    te.append(a.score(Xte, y_test))

plt.plot(prof, tr, "o-", label="entrenament")
plt.plot(prof, te, "s-", label="test")
plt.xlabel("max_depth"); plt.ylabel("precisió"); plt.legend(); plt.show()

d_opt = prof[int(np.argmax(te))]
print(pd.DataFrame({"train": tr, "test": te}, index=prof).round(4))
print("Profunditat més petita amb el millor test:", d_opt)

> **Resultat.** La precisió puja molt de pressa: amb profunditat 1 ja dona 0,883 de test, a profunditat 5 0,999, i a profunditat 7 arriba a 1,0 en test (i en entrenament). Que surti un 100 % no és cap error de codi: aquest dataset és quasi determinista. La profunditat petita interessa perquè l'arbre s'ha de poder explicar (i fer servir) al bosc.

### 2.3 · La teoria, al codi: el Gini del primer tall

Entrena un arbre de profunditat 3. Calcula **a mà** (amb `y_train` i les proporcions de classe) la impuresa de Gini del node arrel i compara-la amb la que guarda scikit-learn. Després, mira quina columna fa el primer tall, separa tu mateix les files d'entrenament en els dos fills i calcula'n el Gini de cadascun i el **Gini ponderat**. Quant baixa la impuresa amb aquest tall?

In [ ]:
def gini(etiquetes):
    p = pd.Series(etiquetes).value_counts(normalize=True).values
    return 1 - np.sum(p ** 2)

arbre = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr, y_train)
t = arbre.tree_

gini_arrel = gini(y_train)
print("Gini arrel a mà:", round(gini_arrel, 6), "| scikit-learn:", round(t.impurity[0], 6))
assert np.isclose(gini_arrel, t.impurity[0])

col, llindar = t.feature[0], t.threshold[0]
print("Primer tall:", noms[col], "<=", llindar)
esq = Xtr[:, col] <= llindar
g_e, g_d = gini(y_train[esq]), gini(y_train[~esq])
fe, fd = t.children_left[0], t.children_right[0]
assert np.isclose(g_e, t.impurity[fe]) and np.isclose(g_d, t.impurity[fd])
assert esq.sum() + (~esq).sum() == t.n_node_samples[0]

ponderat = (esq.sum() * g_e + (~esq).sum() * g_d) / len(y_train)
print(f"Gini fills: {g_e:.4f} i {g_d:.4f} | ponderat: {ponderat:.4f} | baixa: {gini_arrel - ponderat:.4f}")

> **Resultat.** El Gini de l'arrel és 0,4994 (gairebé el màxim de 0,5, perquè les classes estan quasi equilibrades). El primer tall es fa sobre una columna d'olor (`odor_n`, «sense olor»), i el Gini ponderat dels fills (0,2840 i 0,0681) és 0,1897: un sol tall treu el 62 % del desordre.

### 2.4 · I el kNN?

Amb dades codificades en 0/1, què mesura la distància euclidiana entre dos bolets? Comprova-ho amb dues files d'entrenament i compara-ho amb el nombre de trets en què difereixen. Després entrena un kNN amb uns quants valors de `k` i compara'l amb l'arbre de l'2.2.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

for i, j in [(0, 1), (2, 3), (10, 500)]:
    d2 = np.sum((Xtr[i] - Xtr[j]) ** 2)
    dif = (X_train.iloc[i] != X_train.iloc[j]).sum()
    print(f"files {i},{j}: distància² = {d2:.0f}, trets diferents = {dif}")
    assert d2 == 2 * dif

for k in [1, 3, 5, 15, 51]:
    knn = KNeighborsClassifier(n_neighbors=k).fit(Xtr, y_train)
    print(f"k={k:2d}  test = {knn.score(Xte, y_test):.4f}")

> **Resultat.** La distància al quadrat és el doble del nombre de trets diferents (una columna passa de 1 a 0 i una altra de 0 a 1). Per això aquí el kNN té tot el sentit: «s'assembla» vol dir «comparteix trets». El kNN també arriba a 1,0 amb k petita i baixa una mica amb k gran (0,9985 amb k=15, 0,9966 amb k=51). Funciona, però no explica res a la persona del bosc, i l'arbre sí.

### 2.5 · Quan s'equivoca, com s'equivoca

Tria el model que portaries a l'app (amb el criteri de l'associació: ha de ser fàcil de seguir). Treu-ne la matriu de confusió de test i, **a partir de les quatre caselles**, calcula a mà el *recall* dels verinosos. Compara'l amb `recall_score`. Quin dels dos errors és el perillós aquí, i quants n'hi ha?

In [ ]:
from sklearn.metrics import confusion_matrix, recall_score

for d in [3, d_opt]:
    model = DecisionTreeClassifier(max_depth=d, random_state=42).fit(Xtr, y_train)
    pred = model.predict(Xte)
    cm = confusion_matrix(y_test, pred, labels=["e", "p"])
    assert cm.sum() == len(y_test) and cm[1].sum() == (y_test == "p").sum()
    recall_p = cm[1, 1] / cm[1].sum()
    assert np.isclose(recall_p, recall_score(y_test, pred, pos_label="p"))
    print(f"max_depth={d}\n{cm}\n  recall verinosos = {recall_p:.4f}  | verinosos dits comestibles: {cm[1, 0]}")

> **Resultat.** L'error perillós és el **fals negatiu** (verinós dit comestible). Amb profunditat 3, l'arbre en té 3 a test (*recall* de 0,9969, i 32 falsos positius) i és molt llegible; amb la profunditat de l'2.2 (7) en té 0. Aquí hi ha la discussió de fons: un arbre que es pot explicar però que deixa passar 3 verinosos de 979 no és acceptable en una app. Un bon raonament és triar la profunditat petita que dona zero falsos negatius a test, i dir que «zero a test» no és «zero al bosc».

## Nivell 3 · Més enllà

### 3.1 · Només tres preguntes

L'app només pot fer **tres preguntes** a qui té el bolet a la mà. Quins tres trets triaries? Justifica-ho amb el model (no amb la intuïció) i mesura quant perds respecte de fer servir els 22 trets. Fixa't que les importàncies són per a cada columna 0/1, no per a cada tret.

In [ ]:
model = DecisionTreeClassifier(max_depth=d_opt, random_state=42).fit(Xtr, y_train)
imp = pd.Series(model.feature_importances_, index=noms)
per_tret = imp.groupby(lambda s: s.rsplit("_", 1)[0]).sum().sort_values(ascending=False)
print(per_tret.head(6).round(3))

tres = list(per_tret.index[:3])
o3 = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
a3 = DecisionTreeClassifier(random_state=42).fit(o3.fit_transform(X_train[tres]), y_train)
p3 = a3.predict(o3.transform(X_test[tres]))
cm3 = confusion_matrix(y_test, p3, labels=["e", "p"])
print("Tres trets:", tres, "| precisió:", round((p3 == y_test).mean(), 4), "| verinosos dits comestibles:", cm3[1, 0])

> **Resultat.** Les importàncies agregades per tret posen l'olor (0,664) molt per sobre de la resta (`stalk-root` 0,175, `stalk-surface-below-ring` 0,099). Amb aquests tres trets la precisió és 0,9921 i reapareixen 16 falsos negatius: compareu el preu en bolets verinosos, no en punts de precisió. Una bona resposta també diu que els tres trets més importants per a l'arbre complet no són necessàriament els tres millors per separat.

### 3.2 · Amb poques dades

Una altra associació només té 60 bolets descrits. Entrena l'arbre i el kNN amb **només un 1 %** de l'entrenament (unes 60 files, estratificades) i avalua'ls amb el mateix test. Repeteix-ho amb unes quantes llavors. Quin aguanta millor? És el mateix guanyador que amb totes les dades?

In [ ]:
res = []
for s in range(10):
    Xp, _, yp, _ = train_test_split(Xtr, y_train, train_size=0.01, stratify=y_train, random_state=s)
    a = DecisionTreeClassifier(random_state=42).fit(Xp, yp).score(Xte, y_test)
    k = KNeighborsClassifier(n_neighbors=1).fit(Xp, yp).score(Xte, y_test)
    res.append((a, k))
res = pd.DataFrame(res, columns=["arbre", "kNN k=1"])
print("files d'entrenament:", len(yp))
print(res.agg(["mean", "std"]).round(3))

> **Resultat.** Amb 60 files cap dels dos arriba al 100 %: arbre 0,954 ± 0,018 i kNN 0,957 ± 0,011 sobre 10 llavors, pràcticament empatats.

## Conclusió

Sí que es pot decidir amb un model: els 22 trets separen les dues classes gairebé del tot. Un
arbre de profunditat 5 ja encerta el 99,9 % del test i amb profunditat 7 arriba al 100 %; el kNN
també hi arriba, però no s'explica a ningú. El primer tall és l'olor (`odor_n`, «sense olor») i treu
el 62 % de la impuresa de l'arrel (Gini 0,4994 → 0,1897). L'error perillós és dir comestible un
bolet verinós: l'arbre de profunditat 3 en fa 3 al test (*recall* 0,9969), i en una app això no és
acceptable. Cal la profunditat més petita que en faci zero, i dir clar que zero al test no és zero
al bosc.